# 02 — Feature Engineering

Leakage-safe lags, rolling stats, weather, calendar features, and the binary episode target.

**Rule:** features use only information available at time *t*; the target uses *t+1…t+24*.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import load_config, project_path
from src.data.loader import load_site
from src.data.preprocessing import preprocess_site
from src.features.targets import create_target_from_config, target_prevalence
from src.features.engineering import build_feature_matrix, feature_groups

cfg = load_config()
clean = preprocess_site(load_site(), cfg)
labeled = create_target_from_config(clean, cfg)
feat_df, feature_cols = build_feature_matrix(labeled, cfg)
print("n=", len(feat_df), "p=", len(feature_cols))
print(target_prevalence(feat_df["y_episode"]))
groups = feature_groups(feature_cols)
{k: len(v) for k, v in groups.items()}

In [ ]:
import pandas as pd
rows = []
inv = {}
for g, cols in feature_groups(feature_cols).items():
    for c in cols:
        inv[c] = g
for c in feature_cols:
    rows.append({"feature": c, "group": inv.get(c, "other")})
out = project_path("reports/tables/feature_dictionary.csv")
pd.DataFrame(rows).to_csv(out, index=False)
pd.DataFrame(rows).groupby("group").size()